El modelo de **Fama-French-Carhart (FFC)** extiende el CAPM con tres factores adicionales que capturan anomalías documentadas en los mercados: el efecto tamaño (SMB), el efecto valor (HML) y el momentum (MOM). Estimamos las cargas factoriales de cada acción y comparamos los retornos esperados bajo CAPM y FFC.

In [1]:
#| label: setup-ffc
#| code-fold: true
#| code-summary: "Datos y descarga de factores Fama-French"

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import yfinance as yf
import statsmodels.api as sm
from pandas_datareader import data as pdr

plt.rcParams.update({'figure.facecolor': 'white', 'axes.facecolor': 'white',
                     'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 11})

TICKERS = ['AAPL','MSFT','AMZN','GOOGL','META','JPM','BAC','GS','WFC','MS',
           'JNJ','PFE','UNH','MRK','ABBV','XOM','CVX','KO','PG','WMT']

# Retornos de acciones
raw = yf.download(TICKERS, start='2015-01-01', end='2024-12-31',
                  interval='1mo', auto_adjust=True, progress=False)['Close']
returns = raw.dropna(axis=1, thresh=int(0.9*len(raw))).pct_change().dropna()
tickers_avail = returns.columns.tolist()

# Factores Fama-French-Carhart (4 factores) desde Kenneth French Data Library
ff3 = pdr.get_data_famafrench('F-F_Research_Data_Factors', start='2015-01', end='2024-12')[0]
ff4 = pdr.get_data_famafrench('F-F_Momentum_Factor',       start='2015-01', end='2024-12')[0]

factors = ff3.join(ff4, how='inner').rename(columns={'Mkt-RF': 'MKT', 'Mom': 'MOM'})
factors = factors / 100   # están en porcentaje
factors.index = pd.to_datetime(
    factors.index.astype(str).str[:4] + '-' + factors.index.astype(str).str[4:]
)
factors.index = factors.index + pd.offsets.MonthEnd(0)

# Alinear fechas
returns.index = returns.index + pd.offsets.MonthEnd(0)
df = returns.join(factors, how='inner')

print(f"Observaciones alineadas: {len(df)}")
print(f"Factores disponibles: {list(factors.columns)}")
df[list(factors.columns)].describe().round(4)

Observaciones alineadas: 119
Factores disponibles: ['MKT', 'SMB', 'HML', 'RF', 'MOM']


,MKT,SMB,HML,RF,MOM
count,119.0000,119.0000,119.0000,119.0000,119.0000
mean,0.0102,-0.0013,-0.0016,0.0014,0.0017
std,0.0459,0.0280,0.0379,0.0016,0.0404
min,-0.1337,-0.0593,-0.1383,0.0000,-0.1621
25%,-0.0155,-0.0232,-0.0215,0.0001,-0.0223
50%,0.0134,-0.0018,-0.0050,0.0009,0.0045
75%,0.0344,0.0154,0.0176,0.0020,0.0292
max,0.1360,0.0714,0.1286,0.0048,0.0997


### Regresiones de cuatro factores

Para cada acción estimamos por MCO:

$$z_{j,t} = \alpha_j + \beta_{jm} \cdot MKT_t + \beta_{js} \cdot SMB_t + \beta_{jv} \cdot HML_t + \beta_{jmom} \cdot MOM_t + \varepsilon_{j,t}$$

Donde:
- **MKT** = retorno de mercado en exceso sobre Rf
- **SMB** = Small Minus Big (premio por tamaño)
- **HML** = High Minus Low (premio por valor / Book-to-Market)
- **MOM** = momentum (ganadores − perdedores últimos 12 meses)

In [2]:
#| label: ffc-betas

factor_cols = ['MKT', 'SMB', 'HML', 'MOM']
X = sm.add_constant(df[factor_cols])

ffc_results = []
for t in tickers_avail:
    z_j = df[t] - df['RF']
    model = sm.OLS(z_j, X).fit()
    sig = {f: '*' if model.pvalues[f] < 0.05 else '' for f in factor_cols}
    ffc_results.append({
        'Ticker':   t,
        'Alpha':    round(model.params['const'], 5),
        'β_MKT':    round(model.params['MKT'], 3),
        'β_SMB':    round(model.params['SMB'], 3),
        'β_HML':    round(model.params['HML'], 3),
        'β_MOM':    round(model.params['MOM'], 3),
        'R²':       round(model.rsquared, 3),
        'sig_SMB':  sig['SMB'],
        'sig_HML':  sig['HML'],
        'sig_MOM':  sig['MOM'],
    })

ffc_df = pd.DataFrame(ffc_results).set_index('Ticker')

print("Factor loadings (β_SMB > 0: empresa pequeña; β_HML > 0: valor; β_MOM > 0: momentum)")
print("* = significativo al 5%")
print()
print(f"Mayor exposición a SMB: {ffc_df['β_SMB'].idxmax()} ({ffc_df['β_SMB'].max():.3f})  → empresa pequeña")
print(f"Menor exposición a SMB: {ffc_df['β_SMB'].idxmin()} ({ffc_df['β_SMB'].min():.3f})  → empresa grande")
print(f"Mayor exposición a HML: {ffc_df['β_HML'].idxmax()} ({ffc_df['β_HML'].max():.3f})  → Q de Tobin baja")
print(f"Menor exposición a HML: {ffc_df['β_HML'].idxmin()} ({ffc_df['β_HML'].min():.3f})  → Q de Tobin alta")
print(f"Mayor exposición a MOM: {ffc_df['β_MOM'].idxmax()} ({ffc_df['β_MOM'].max():.3f})  → ganador reciente")
print(f"Menor exposición a MOM: {ffc_df['β_MOM'].idxmin()} ({ffc_df['β_MOM'].min():.3f})  → perdedor reciente")
print()
ffc_df[['Alpha', 'β_MKT', 'β_SMB', 'β_HML', 'β_MOM', 'R²']].sort_values('β_MKT', ascending=False)

Factor loadings (β_SMB > 0: empresa pequeña; β_HML > 0: valor; β_MOM > 0: momentum)
* = significativo al 5%

Mayor exposición a SMB: GS (0.287)  → empresa pequeña
Menor exposición a SMB: KO (-0.725)  → empresa grande
Mayor exposición a HML: BAC (1.022)  → Q de Tobin baja
Menor exposición a HML: AMZN (-0.967)  → Q de Tobin alta
Mayor exposición a MOM: UNH (0.239)  → ganador reciente
Menor exposición a MOM: META (-0.479)  → perdedor reciente



,Alpha,β_MKT,β_SMB,β_HML,β_MOM,R²
Ticker,,,,,,
BAC,0.00079,1.371,0.125,1.022,0.019,0.771
AMZN,0.00926,1.263,-0.259,-0.967,-0.109,0.582
GS,0.00243,1.261,0.287,0.718,-0.045,0.693
AAPL,0.00745,1.221,-0.164,-0.522,0.009,0.528
MS,0.00511,1.198,0.284,0.634,-0.271,0.688
JPM,0.00593,1.115,0.166,0.853,0.132,0.737
WFC,-0.00171,1.041,0.098,0.956,-0.135,0.605
META,0.00905,1.035,-0.691,-0.716,-0.479,0.360
MSFT,0.00992,1.009,-0.685,-0.451,-0.060,0.617


### Retornos esperados: CAPM vs FFC

El retorno esperado bajo FFC usa solo los coeficientes significativos:

$$E(r_j)_{FFC} = R_f + \hat\beta_{jm}\bar{z}_m + \hat\beta_{js}\bar{z}_s + \hat\beta_{jv}\bar{z}_v + \hat\beta_{jmom}\bar{z}_{mom}$$

Comparamos con el CAPM de Sharpe:

$$E(r_j)_{CAPM} = R_f + \hat\beta_{jm}\bar{z}_m$$

In [3]:
#| label: retornos-comparacion
#| code-fold: true
#| fig-cap: "Retornos esperados anualizados: CAPM vs Fama-French-Carhart"

RF_m = df['RF'].mean()
factor_means = {f: df[f].mean() for f in factor_cols}

comp = []
for t in tickers_avail:
    r_j_series = df[t] - df['RF']
    X_ = sm.add_constant(df[factor_cols])
    m = sm.OLS(r_j_series, X_).fit()

    # Solo factores significativos para FFC
    e_capm = (RF_m + m.params['MKT'] * factor_means['MKT']) * 12
    e_ffc  = RF_m * 12
    for f in factor_cols:
        if m.pvalues[f] < 0.05:
            e_ffc += m.params[f] * factor_means[f] * 12

    comp.append({'Ticker': t, 'CAPM (anual)': e_capm * 100, 'FFC (anual)': e_ffc * 100,
                 'Diferencia |%|': abs(e_capm - e_ffc) * 100})

comp_df = pd.DataFrame(comp).set_index('Ticker').sort_values('Diferencia |%|', ascending=False)

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(comp_df))
w = 0.35
ax.bar(x - w/2, comp_df['CAPM (anual)'], w, label='CAPM', color='#2563eb', alpha=0.85)
ax.bar(x + w/2, comp_df['FFC (anual)'],  w, label='FFC (4 factores)', color='#f59e0b', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(comp_df.index, rotation=45, ha='right')
ax.set_ylabel('Retorno esperado anual (%)')
ax.set_title('Retorno esperado: CAPM vs Fama-French-Carhart (2015–2024)', fontsize=13, fontweight='bold')
ax.legend()
ax.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

max_diff = comp_df.iloc[0]
print(f"Mayor diferencia entre CAPM y FFC: {max_diff.name} ({max_diff['Diferencia |%|']:.2f} pp)")
print(f"\nDiferencia promedio entre modelos: {comp_df['Diferencia |%|'].mean():.2f} pp")
print()
comp_df.round(3)

Mayor diferencia entre CAPM y FFC: BAC (1.92 pp)

Diferencia promedio entre modelos: 1.15 pp



,CAPM (anual),FFC (anual),Diferencia |%|
Ticker,,,
BAC,18.397,16.480,1.917
MSFT,13.989,15.866,1.877
XOM,11.940,10.086,1.854
AMZN,17.089,18.902,1.813
WFC,14.383,12.591,1.792
MS,16.294,14.559,1.735
CVX,13.469,11.824,1.645
JPM,15.287,13.688,1.599
GOOGL,13.586,15.077,1.492
